# Contar usuarios por género

## Qué hace

Muestra los tres totales calculados por el modelo local.

## Qué pasa en la base de datos

Solo consulta nombres; no guarda el género.

## Qué acción usa

`await usersGenderCount()`.

## Anotaciones

Necesita Ollama con llama3.2. Sin usuarios devuelve tres ceros. Si falla, muestra Could not classify the users. Es una estimación por nombre.

Selecciona la celda de abajo y pulsa **▶** o **Shift+Enter**. No necesitas
escribir código. El resultado aparece debajo en una tabla con encabezados cian.
Los datos de demostración ya están preparados en el documento.


In [ ]:
import json
import subprocess
from pathlib import Path

location = Path.cwd().resolve()
app = None
for directory in (location, *location.parents):
    candidate = directory / "staff/jimena-duran/first-app"
    source = candidate / "src"
    markers = ("logic/create-user.js", "queries/users.js", "db/connection.js")
    if (candidate / "package.json").is_file() and all((source / marker).is_file() for marker in markers):
        app = source
        break
    if (directory / ".git").exists():
        break
if app is None:
    raise SystemExit("No encuentro staff/jimena-duran/first-app/src. Abre Jupyter desde first-app, src o una subcarpeta de esta app.")

code = r"""
import { usersGenderCount } from "./logic/users-gender-count.js";


try {
  
  const result = await usersGenderCount();
  console.log(JSON.stringify(result === undefined ? null : result));
} catch (error) {
  console.error(error.message);
  process.exitCode = 1;
}
"""

try:
    completed = subprocess.run(
        ["node", "--input-type=module", "--eval", code],
        cwd=app,
        capture_output=True,
        encoding="utf-8",
    )
except OSError:
    raise SystemExit("No encuentro Node.js. Revisa la preparación de la app.")

if completed.returncode != 0:
    print(completed.stderr.strip())
else:
    result = json.loads(completed.stdout)
    headers = ['women', 'men', 'genderUndefined']
    rows = [[result[field] for field in headers]]
    # Tabla de texto con encabezados cian, sin librerías adicionales.
    rows = [[str(value).replace("\n", " ").replace("\r", " ").replace("\x1b", "") for value in row] for row in rows]
    widths = [max(len(header), *(len(row[index]) for row in rows)) if rows else len(header) for index, header in enumerate(headers)]
    print("┌" + "┬".join("─" * (width + 2) for width in widths) + "┐")
    print("│ " + " │ ".join("\x1b[1;36m" + header.ljust(width) + "\x1b[0m" for header, width in zip(headers, widths)) + " │")
    print("├" + "┼".join("─" * (width + 2) for width in widths) + "┤")
    for row in rows:
        print("│ " + " │ ".join(value.ljust(width) for value, width in zip(row, widths)) + " │")
    print("└" + "┴".join("─" * (width + 2) for width in widths) + "┘")
    if not rows:
        print("No hay filas.")
